# 06 — Modelos pareados de demanda (clima / atraso)

Treina e exporta os **dois modelos binários pareados** que o narrador (`07_narrador_natural`) consome, além do **prior prospectivo de atraso** (climatologia + reputação por `dia_semana × hora`). É a porta da seção de PRODUÇÃO do notebook `03_ANALISE/05_6_FeatureEng_Individual_vs_Agrupado`, adaptada à base do pacote.

**Base de dados (a mesma do pacote):**
- eventos de corrida — `driver_trips` (amostra estratificada no DuckDB; `_tmp_big` por padrão)
- clima — `data/raw/meteo.parquet` (SBPA)
- voos — `data/raw/flights_anac.parquet`

**Saídas** (em `Work/narrador_artefatos/`): `modelo_par_clima.joblib` + `_meta.json`, `modelo_par_atraso.joblib` + `_meta.json`, `climatologia_atraso_sbpa.csv`, `reputacao_slot_sbpa.csv`.

> A lógica vive em `tese_pipeline.narrador.treina_pareados` e `tese_pipeline.narrador.features` — o mesmo módulo de features usado pelo narrador, garantindo consistência treino×inferência (sem *train/serve skew*).

In [ ]:
# Garante que o diretório de trabalho é a raiz do pacote (onde estão data/, Work/, params.yaml).
import os
from pathlib import Path

while not Path("params.yaml").exists() and Path.cwd() != Path.cwd().parent:
    os.chdir("..")
print("Diretório de trabalho:", Path.cwd())

In [ ]:
# Treina os dois modelos pareados e exporta todos os artefatos.
# Parâmetros (fonte dos eventos, janela, pesos, Optuna) ficam no topo de treina_pareados.py.
from tese_pipeline.narrador import treina_pareados as tp

tp.main()

In [ ]:
# Confere os artefatos gerados e um resumo dos metadados.
import json
from pathlib import Path

ART = Path("Work/narrador_artefatos")
print("Artefatos:", sorted(p.name for p in ART.glob("*")))
for nome in ("clima", "atraso"):
    m = json.loads((ART / f"modelo_par_{nome}_meta.json").read_text(encoding="utf-8"))
    print(f"\n[{nome}] janela={m['janela']} | features={len(m['features_lgbm'])} "
          f"| macro_f1={m['macro_f1_bal']} | pr_auc={m['pr_auc']}")
    print("  climatologia_atraso:", m["climatologia_atraso"])